# 01 - Data understanding (STATS19)
Reads the reports written by `inspect_data.py` and `prepare_data.py`. Nothing here is hard-coded; every number comes from the real files you downloaded.

**Questions this notebook answers:** What is the dataset (source, licence, size, period)? Which columns exist? How much is missing/unknown? Are there duplicates or outliers? How imbalanced is the target? Which columns are excluded to prevent leakage, and why?

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
ROOT = Path(os.environ.get("ROADRISK_ROOT") or Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(ROOT))
from ml.config import get_settings
from ml.feature_config import CLASS_NAMES
cfg = get_settings()
pd.set_option("display.max_columns", 50, "display.width", 140)
def load_report(name):
    p = cfg.reports_dir / name
    if not p.exists():
        raise FileNotFoundError(f"{p} not found - run the pipeline scripts first (see README).")
    return json.loads(p.read_text(encoding="utf-8"))
COLORS = {"Slight": "#5b8def", "Serious": "#f2a33a", "Fatal": "#d64545"}

## 1. Source, licence, size, period

In [ ]:
dr = load_report("data_report.json")
d = dr["dataset"]
print("Source  :", d["name"]); print("Licence :", d["licence"])
print("Years requested:", d["years_requested"], "| used:", d["years_used"], "| missing:", d["years_missing"])
print(f"Rows: {dr['n_rows_loaded']:,}  Columns loaded: {dr['n_columns_loaded']}")
pd.DataFrame(d["files"])[["year", "file", "size_mb", "rows_loaded", "columns_in_file"]]

## 2. Raw schema (all columns in the files)

In [ ]:
sr = load_report("schema_report.json")
first = sorted(sr["files"])[0]
cols = pd.DataFrame(sr["files"][first]["columns"])[["column", "dtype_inferred", "pct_missing", "n_unique", "sample_values", "used_by_pipeline"]]
print("Columns differing between years:", sr["column_differences_between_years"]["not_in_all_years"])
cols

## 3. Missing values, special codes, duplicates

In [ ]:
miss = pd.DataFrame(dr["columns"]).sort_values("pct_missing", ascending=False)
display(miss.head(15))
print("Duplicates:", dr["duplicates"])
pd.DataFrame({c: {"% unknown/missing": v["pct_unknown_or_missing"]} for c, v in dr["special_code_analysis"].items()}).T.sort_values("% unknown/missing", ascending=False)

## 4. Outliers and suspicious values

In [ ]:
print("Outliers (IQR rule):"); display(pd.DataFrame(dr["outlier_analysis_iqr"]).T)
print("Handling:", dr["outlier_handling"])
print("Suspicious values:"); display(pd.Series(dr["suspicious_values"], name="count").to_frame())

## 5. Target distribution and imbalance

In [ ]:
td = load_report("target_distribution.json")
print("Raw codes observed:", td["raw_values_observed"], "-> mapping", td["mapping_used"])
print("Sanity check:", td["sanity_check_note"])
s = pd.Series(td["class_counts"]).reindex(CLASS_NAMES)
print((100 * s / s.sum()).round(2).to_string(), "\nimbalance ratio:", td["imbalance_ratio_majority_to_minority"])
ax = s.plot.bar(color=[COLORS[c] for c in s.index], figsize=(5, 3), title="Collision severity (rows)"); ax.set_yscale("log"); plt.show()

## 6. Leakage control: features used vs excluded

In [ ]:
print("Excluded features and reasons:")
pd.Series(dr["excluded_features"], name="reason").to_frame()

In [ ]:
lr = cfg.reports_dir / "leakage_report.json"
if lr.exists():
    L = json.loads(lr.read_text()); print("Leakage audit passed:", L["passed"]); print(L["checks"])
    display(pd.Series(L["single_feature_cramers_v_with_target"], name="Cramer's V vs target").sort_values(ascending=False).head(10).to_frame())
else:
    print("leakage_report.json is created by train_models.py (run it first).")

## Your observations
_Write your own conclusions here after running the notebook on your data._